1 — Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

2 — Set paths

In [2]:
TRAIN_FILE = Path("train.parquet")
VAL_FILE = Path("validation.parquet")
TEST_FILE = Path("test.parquet")

3 — Check that they exist

In [3]:
for file_path in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    print(file_path, "->", file_path.exists())

train.parquet -> True
validation.parquet -> True
test.parquet -> True


4 — Inspect the columns

In [4]:
train_sample = pd.read_parquet(TRAIN_FILE)

print(train_sample.head())
print("\nColumns:")
print(train_sample.columns.tolist())
print("\nShape:")
print(train_sample.shape)

                 event_time event_type  product_id          category_id  \
0 2019-10-01 00:00:00+00:00       view    44600062  2103807459595387724   
1 2019-10-01 00:00:00+00:00       view     3900821  2053013552326770905   
2 2019-10-01 00:00:01+00:00       view    17200506  2053013559792632471   
3 2019-10-01 00:00:01+00:00       view     1307067  2053013558920217191   
4 2019-10-01 00:00:04+00:00       view     1004237  2053013555631882655   

                         category_code     brand        price    user_id  \
0                                 None  shiseido    35.790001  541312140   
1  appliances.environment.water_heater      aqua    33.200001  554748717   
2           furniture.living_room.sofa      None   543.099976  519107250   
3                   computers.notebook    lenovo   251.740005  550050854   
4               electronics.smartphone     apple  1081.979980  535871217   

                           user_session  
0  72d76fde-8bb3-4e00-8c23-a032dfed738c  
1  9333d

5 — Aggregation function

In [5]:
def aggregate_user_product_interactions(
    file_path,
    columns=None
):
    
    if columns is None:
        columns = [
            "event_type",
            "product_id",
            "user_id"
        ]
    
    df = pd.read_parquet(
        file_path,
        columns=columns
    )
    
    # Remove rows with missing essential IDs
    df = df.dropna(
        subset=["user_id", "product_id", "event_type"]
    )
    
    # Convert IDs to strings for consistency
    df["user_id"] = df["user_id"].astype(str)
    df["product_id"] = df["product_id"].astype(str)
    
    # Create event indicators
    df["click"] = (
        df["event_type"] == "view"
    ).astype("int8")
    
    df["cart"] = (
        df["event_type"] == "cart"
    ).astype("int8")
    
    df["purchase"] = (
        df["event_type"] == "purchase"
    ).astype("int8")
    
    # Aggregate user-product interactions
    interactions = (
        df.groupby(
            ["user_id", "product_id"],
            as_index=False
        )
        .agg(
            clicks=("click", "sum"),
            cart=("cart", "sum"),
            purchase=("purchase", "sum")
        )
    )
    
    return interactions

6 — Aggregate TRAIN

In [6]:
train_interactions = aggregate_user_product_interactions(
    TRAIN_FILE
)

print("Train interaction shape:")
print(train_interactions.shape)

print("\nFirst rows:")
print(train_interactions.head())

Train interaction shape:
(16275394, 5)

First rows:
     user_id product_id  clicks  cart  purchase
0  183503497   22200103       1     0         0
1  184265397   27400002       2     0         0
2  184265397    6902133       2     0         0
3  184265397    6902303       2     0         0
4  195082191    4804056       1     0         0


7 — Aggregate VALIDATION

In [7]:
validation_interactions = aggregate_user_product_interactions(
    VAL_FILE
)

print("Validation interaction shape:")
print(validation_interactions.shape)

print("\nFirst rows:")
print(validation_interactions.head())

Validation interaction shape:
(4036632, 5)

First rows:
     user_id product_id  clicks  cart  purchase
0  209714031   26900028       3     0         0
1  209714031   44500029       1     0         0
2  224520397    8800028       1     0         0
3  226242984    1005135       1     0         0
4  226242984    1802045       1     0         0


8 — Aggregate TEST

In [8]:
test_interactions = aggregate_user_product_interactions(
    TEST_FILE
)

print("Test interaction shape:")
print(test_interactions.shape)

print("\nFirst rows:")
print(test_interactions.head())

Test interaction shape:
(3727093, 5)

First rows:
     user_id product_id  clicks  cart  purchase
0  209714031    5500007       4     0         0
1  209714031    5500032       2     0         0
2  209714031    5500097       1     0         0
3  209714031    5500107       2     0         0
4  209714031    5500108       2     0         0


9 — Save them

In [9]:
TRAIN_OUTPUT = "DiscountMATE_Feature2_REES46_train_user_product_interactions.parquet"
VAL_OUTPUT = "DiscountMATE_Feature2_REES46_validation_user_product_interactions.parquet"
TEST_OUTPUT = "DiscountMATE_Feature2_REES46_test_user_product_interactions.parquet"

train_interactions.to_parquet(
    TRAIN_OUTPUT,
    index=False
)

validation_interactions.to_parquet(
    VAL_OUTPUT,
    index=False
)

test_interactions.to_parquet(
    TEST_OUTPUT,
    index=False
)

print("All three interaction datasets saved successfully.")

All three interaction datasets saved successfully.


10 — Check the datasets

In [10]:
print("TRAIN")
print(train_interactions.shape)

print("\nVALIDATION")
print(validation_interactions.shape)

print("\nTEST")
print(test_interactions.shape)

TRAIN
(16275394, 5)

VALIDATION
(4036632, 5)

TEST
(3727093, 5)


In [11]:
print("Train purchases:",
      train_interactions["purchase"].sum())

print("Validation purchases:",
      validation_interactions["purchase"].sum())

print("Test purchases:",
      test_interactions["purchase"].sum())

Train purchases: 516347
Validation purchases: 120697
Test purchases: 105805


In [12]:
print("Train carts:",
      train_interactions["cart"].sum())

print("Validation carts:",
      validation_interactions["cart"].sum())

print("Test carts:",
      test_interactions["cart"].sum())

Train carts: 642499
Validation carts: 180674
Test carts: 103343


11 — Check the user/product coverage

In [13]:
train_users = set(
    train_interactions["user_id"]
)

val_users = set(
    validation_interactions["user_id"]
)

test_users = set(
    test_interactions["user_id"]
)

print("Train users:", len(train_users))
print("Validation users:", len(val_users))
print("Test users:", len(test_users))

print(
    "Validation cold-start users:",
    len(val_users - train_users)
)

print(
    "Test users not seen in train:",
    len(test_users - train_users)
)

Train users: 2312200
Validation users: 839354
Test users: 764234
Validation cold-start users: 391205
Test users not seen in train: 378996


To convert parquet to csv files

In [14]:
import pandas as pd

# Convert Train
pd.read_parquet(
    "DiscountMATE_Feature2_REES46_train_user_product_interactions.parquet"
).to_csv(
    "DiscountMATE_Feature2_REES46_train_user_product_interactions.csv",
    index=False
)

# Convert Validation
pd.read_parquet(
    "DiscountMATE_Feature2_REES46_validation_user_product_interactions.parquet"
).to_csv(
    "DiscountMATE_Feature2_REES46_validation_user_product_interactions.csv",
    index=False
)

# Convert Test
pd.read_parquet(
    "DiscountMATE_Feature2_REES46_test_user_product_interactions.parquet"
).to_csv(
    "DiscountMATE_Feature2_REES46_test_user_product_interactions.csv",
    index=False
)

print("✅ All 3 Parquet files converted to CSV successfully!")

✅ All 3 Parquet files converted to CSV successfully!
